## 🕵️‍♂️ Case File #10: The Islamabad Tech Hub Energy Grid Anomaly Matrix

In [226]:
# import the necessary libraries in the project
import pandas as pd
import numpy as np

# raw log dataset
raw_energy_data = {
    "Reading_Time": ["2026-06-29 09:00", "2026-06-29 10:00", "2026-06-29 11:00", "2026-06-29 09:00", "2026-06-29 10:00", "2026-06-29 11:00", "2026-06-29 09:00", "2026-06-29 10:00"],
    "Facility_Wing": ["Data_Center", "Data_Center", "Data_Center", "Software_Labs", "Software_Labs", "Software_Labs", "Incubator", "Incubator"],
    "Power_Consumption_kW": [450.2, 890.5, 460.1, 120.4, 750.8, 115.2, 45.0, 52.1],
    "Grid_Status": ["Normal", "Normal", "Normal", "Normal", "Fluctuation", "Normal", "Normal", "Normal"]
}

# convert the data into pandas dataframe
df_energy = pd.DataFrame(raw_energy_data)
# check the data
df_energy.sample(3)

,Reading_Time,Facility_Wing,Power_Consumption_kW,Grid_Status
6,2026-06-29 09:00,Incubator,45.0,Normal
5,2026-06-29 11:00,Software_Labs,115.2,Normal
7,2026-06-29 10:00,Incubator,52.1,Normal


In [227]:
# get over view of data
df_energy.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Reading_Time          8 non-null      str    
 1   Facility_Wing         8 non-null      str    
 2   Power_Consumption_kW  8 non-null      float64
 3   Grid_Status           8 non-null      str    
dtypes: float64(1), str(3)
memory usage: 388.0 bytes


#### 1. **Temporal Engineering:** Convert the `Reading_Time` column from a string object type into a proper Pandas `datetime64` object. Once converted, extract just the **Hour** integer into a new standalone column named `Hour`.

In [228]:
# convert the string column into a datetime column using pandas built-in method
df_energy['Reading_Time'] = pd.to_datetime(df_energy['Reading_Time'])

In [229]:
# extract the hour from a Reading_Time column and create a standalone column
df_energy['Hour'] = df_energy['Reading_Time'].dt.hour

#### 2. **Wing-Specific Profiling:** Group the data by `Facility_Wing` and compute a single summary table showing three statistical metrics for each wing's `Power_Consumption_kW:` the **Total**, the **Maximum**, and the **Standard Deviation**.

In [230]:
# group the data by Facility_Wing and create summary table using agg() by 
# performing operations on columns
df_energy.groupby('Facility_Wing').agg(
    total = ('Power_Consumption_kW', 'sum'),
    max = ('Power_Consumption_kW', 'max'),
    std = ('Power_Consumption_kW', 'std')
)

,total,max,std
Facility_Wing,,,
Data_Center,1800.8,890.5,251.398177
Incubator,97.1,52.1,5.020458
Software_Labs,986.4,750.8,365.471969


#### 3. **Statistical Outlier Detection (Z-Score Concept):** Calculate the global mean ($\mu$) and global standard deviation ($\sigma$) of the entire `Power_Consumption_kW` column.
 - A reading is flagged as a **Critical Outlier** if its power value is **more than 1.2 global standard deviations away from the global mean**.
 - Mathematically, filter your dataframe to show rows where:
 - $$\text{Power\_Consumption\_kW} > (\mu + 1.2 \times \sigma) \quad \text{OR} \quad \text{Power\_Consumption\_kW} < (\mu - 1.2 \times \sigma)$$

In [231]:
# calculate the mean by every wing using the groupby method & 
# transform method (remain the same of size of the dataset) 
wing_mean = df_energy.groupby('Facility_Wing')['Power_Consumption_kW'].transform('mean')
wing_std = df_energy.groupby('Facility_Wing')['Power_Consumption_kW'].transform('std')

In [234]:
df_energy

,Reading_Time,Facility_Wing,Power_Consumption_kW,Grid_Status,Hour,Outlier_Flag
0,2026-06-29 09:00:00,Data_Center,450.2,Normal,9,False
1,2026-06-29 10:00:00,Data_Center,890.5,Normal,10,False
2,2026-06-29 11:00:00,Data_Center,460.1,Normal,11,False
3,2026-06-29 09:00:00,Software_Labs,120.4,Normal,9,False
4,2026-06-29 10:00:00,Software_Labs,750.8,Fluctuation,10,False
5,2026-06-29 11:00:00,Software_Labs,115.2,Normal,11,False
6,2026-06-29 09:00:00,Incubator,45.0,Normal,9,False
7,2026-06-29 10:00:00,Incubator,52.1,Normal,10,False


In [235]:
# flag the outliers using this formula
df_energy['Outlier_Flag'] = (df_energy['Power_Consumption_kW'] - wing_mean).abs() > (1.2 * wing_std)

#### 4. **Contextual Isolation:** Isolate all readings where the `Grid_Status` indicates a `"Fluctuation"` OR the extracted `Hour` is peak morning (`9`).

In [236]:
# identify the risk periods for the facility engineeer
df_energy[(df_energy['Grid_Status'] == 'Fluctuation') | (df_energy['Hour'] == 9)]

,Reading_Time,Facility_Wing,Power_Consumption_kW,Grid_Status,Hour,Outlier_Flag
0,2026-06-29 09:00:00,Data_Center,450.2,Normal,9,False
3,2026-06-29 09:00:00,Software_Labs,120.4,Normal,9,False
4,2026-06-29 10:00:00,Software_Labs,750.8,Fluctuation,10,False
6,2026-06-29 09:00:00,Incubator,45.0,Normal,9,False
